# ZELDA wavefront sensor

ZELDA is a phase-mask wavefront sensor.

The optical sequence is

```
entrance pupil
    ↓
aberration
    ↓
pupil → focal plane
    ↓
ZELDA phase mask
    ↓
focal → pupil plane
    ↓
detector
```

The phase mask introduces an optical path difference close to \(\lambda/4\),
corresponding to a \(\pi/2\) phase shift.

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from fiatlux.core.grid import Grid
from fiatlux.core.spectrum import Band, Spectrum
from fiatlux.core.source import PlaneWave
from fiatlux.optics.elements.mask import CircularAperture, Step, ZeldaMask
from fiatlux.optics.propagator import MFTPropagator
from fiatlux.optics.detector import Detector
from fiatlux.system.optical_system import SerialSystem

In [ ]:
D = 1.0
wavelength = 1.65e-6
focal_length = 10.0

N_pupil = 256
N_focal = 256

pupil_grid = Grid(
    nx=N_pupil,
    ny=N_pupil,
    dx=D / N_pupil,
    dy=D / N_pupil,
)

focal_grid = Grid(
    nx=N_focal,
    ny=N_focal,
    dx=focal_length * wavelength / D / 4,
    dy=focal_length * wavelength / D / 4,
)

band = Band(
    central_wavelength=wavelength,
    delta_wavelength=0.0,
    f0=368.0,
)

spectrum = Spectrum(magnitude=0, band=band, samples=1)
source = PlaneWave(spectrum=spectrum)

aperture = CircularAperture(grid=pupil_grid, radius=D / 2)

step = Step(
    grid=pupil_grid,
    piston=wavelength / 20,
)

## Create the focal-plane ZELDA mask

In [ ]:
pupil_to_focal = MFTPropagator(
    focal_length=focal_length,
    output_grid=focal_grid,
)

zelda_radius = focal_length * wavelength / D
zelda_depth = wavelength / 4

zelda_mask = ZeldaMask(
    grid=focal_grid,
    radius=zelda_radius,
    well_depth=zelda_depth,
)

back_to_pupil = MFTPropagator(
    focal_length=focal_length,
    output_grid=pupil_grid,
)

## Create the detector and optical system

In [ ]:
detector = Detector(
    grid=pupil_grid,
    photon_noise=False,
    readout_noise_variance=0,
    dark_current=0,
)

zelda_system = SerialSystem(
    elements=[
        aperture,
        step,
        pupil_to_focal,
        zelda_mask,
        back_to_pupil,
    ]
)

In [ ]:
zelda_result = zelda_system.run(
    source=source,
    detector=detector,
)

## Display the ZELDA pupil image

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(
    detector.image_buffer.cpu(),
    origin="lower",
)

plt.title("ZELDA pupil image")
plt.colorbar(label="Signal")
plt.show()

## Inspect the intermediate focal-plane field

In [ ]:
zelda_focal_field = zelda_result.field_at(zelda_mask)

plt.figure(figsize=(6, 5))
plt.imshow(
    zelda_focal_field.intensity()[0].cpu(),
    origin="lower",
    norm=LogNorm(),
)

plt.title("Field after ZELDA focal-plane mask")
plt.show()